# TFW YOLOv5n-Face 热成像视频运行入口

这个 Notebook 直接调用项目的 Python 主函数，不需要在终端拼接命令。请按顺序运行单元格。它只在最后一个运行单元格中读取 BIN 文件，前面的环境和参数单元格不会访问视频数据。

In [5]:
from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

# 无论从项目根目录还是 Notebook 所在目录启动，都自动定位运行目录。
cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd / 'python_code' / 'thermal_face_baseline',
    *cwd.parents,
]
PROJECT_DIR = next(
    (path for path in candidates if (path / 'run_inference.py').is_file()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('无法定位 thermal_face_baseline/run_inference.py，请从项目目录打开 Notebook。')

PROJECT_DIR = PROJECT_DIR.resolve()
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

try:
    import cv2
    import numpy as np
    import torch
    import run_inference
except ModuleNotFoundError as error:
    raise RuntimeError(
        '当前 Notebook 内核缺少项目依赖。请在右上角选择 '
        'thermal_face_baseline/.venv/Scripts/python.exe 作为内核。'
    ) from error

expected_python = (PROJECT_DIR / '.venv' / 'Scripts' / 'python.exe').resolve()
print(f'项目目录: {PROJECT_DIR}')
print(f'Notebook Python: {sys.executable}')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
print(f'OpenCV: {cv2.__version__} | NumPy: {np.__version__}')
if Path(sys.executable).resolve() != expected_python:
    print(f'提示：推荐在 Notebook 右上角切换到项目环境：{expected_python}')

项目目录: D:\major\IRsegementation\IRrecognize\python_code\thermal_face_baseline
Notebook Python: d:\Program\conda_env\py310\python.exe
PyTorch: 2.11.0+cpu | CUDA: False
OpenCV: 4.13.0 | NumPy: 2.2.6
提示：推荐在 Notebook 右上角切换到项目环境：D:\major\IRsegementation\IRrecognize\python_code\thermal_face_baseline\.venv\Scripts\python.exe


## 运行参数

`FRAME_STEP=1` 保证每个原始帧都进入检测和 AVI；`SHOW_EVERY=5` 只让实时窗口每5帧刷新一次，不会造成 AVI 丢帧。`MAX_FRAMES=0` 表示处理完整视频。

In [ ]:
# ===== 只需要修改这个单元格 =====
BIN_PATH = Path(r'F:\0626\gwe\nir_temp_frame_16uc1.bin')
OUTPUT_DIR = PROJECT_DIR / 'output' / 'bin_full_video_notebook'

# START_FRAME = 0
# END_FRAME = None       # 不包含该帧；None 表示到文件末尾
START_FRAME = 15000
END_FRAME = 15500
# START_FRAME = 12000
# END_FRAME = 12500
FRAME_STEP = 1         # 必须为 1，AVI 才包含所有原始帧
MAX_FRAMES = 0         # 0 表示不限制；调试时可先改为 600
SOURCE_FPS = 30.0      # 请按采集设备的真实帧率修改

SHOW = True            # 实时显示外部 OpenCV 窗口
SHOW_EVERY = 10         # 每5个已检测帧刷新一次窗口，不影响 AVI
SAVE_VIDEO = True
VIDEO_NAME = 'detection_preview.avi'
SAVE_IMAGES = 'auto'   # BIN 的 auto = 只保存没有检测到人脸的帧

DEVICE = 'auto'        # auto / cpu / cuda:0
IMG_SIZE = 800
CONF_THRESHOLD = 0.03  # 候选下限；低于可靠阈值的框必须通过位置和面积门控
RELIABLE_CONF_THRESHOLD = 0.40
MAX_INTERPOLATION_SECONDS = 0.20  # 30 FPS 时最多插值 6 个连续缺失帧的几何位置
IOU_THRESHOLD = 0.50
NORMALIZATION = 'percentile'
LOWER_PERCENTILE = 1.0
UPPER_PERCENTILE = 99.0
LOG_EVERY = 100        # Notebook 每100帧打印一次；漏检帧始终打印

SAVE_ROI_MAT = True       # 导出 MATLAB v7.3 温度 ROI Tensor
SAVE_ROI_PREVIEW = True   # 生成 roi_tracking_preview.avi 供人工核查
ROI_SIZE = 128
SMALL_ROI_SIZE = 64
MAT_CHUNK_FRAMES = 3000

print(f'输入: {BIN_PATH}')
print(f'输出: {OUTPUT_DIR}')
print(f'完整 AVI: frame_step={FRAME_STEP}, source_fps={SOURCE_FPS}')

输入: F:\0718\lgh\nir_temp_frame_16uc1.bin
输出: D:\major\IRsegementation\IRrecognize\python_code\thermal_face_baseline\output\bin_full_video_notebook
完整 AVI: frame_step=1, source_fps=30.0


In [7]:
def build_arguments() -> list[str]:
    args = [
        '--input', str(BIN_PATH),
        '--output', str(OUTPUT_DIR),
        '--start-frame', str(START_FRAME),
        '--frame-step', str(FRAME_STEP),
        '--max-frames', str(MAX_FRAMES),
        '--source-fps', str(SOURCE_FPS),
        '--show-every', str(SHOW_EVERY),
        '--save-images', SAVE_IMAGES,
        '--video-name', VIDEO_NAME,
        '--device', DEVICE,
        '--img-size', str(IMG_SIZE),
        '--conf-thres', str(CONF_THRESHOLD),
        '--reliable-conf-thres',str(RELIABLE_CONF_THRESHOLD),
        '--max-interpolation-seconds', str(MAX_INTERPOLATION_SECONDS),
        '--iou-thres', str(IOU_THRESHOLD),
        '--normalization', NORMALIZATION,
        '--lower-percentile', str(LOWER_PERCENTILE),
        '--upper-percentile', str(UPPER_PERCENTILE),
        '--log-every', str(LOG_EVERY),
        '--roi-size', str(ROI_SIZE),
        '--small-roi-size', str(SMALL_ROI_SIZE),
        '--mat-chunk-frames', str(MAT_CHUNK_FRAMES),
    ]
    if END_FRAME is not None:
        args.extend(['--end-frame', str(END_FRAME)])
    if SHOW:
        args.append('--show')
    if SAVE_VIDEO:
        args.append('--save-video')
    if SAVE_ROI_MAT:
        args.append('--save-roi-mat')
    if SAVE_ROI_PREVIEW:
        args.append('--save-roi-preview')
    return args


def run_project() -> Path:
    # 重新加载模块，使修改 run_inference.py 后无需重启 Notebook 内核。
    importlib.reload(run_inference)
    argv = build_arguments()
    print('开始运行 TFW 热成像人脸检测……')
    print(f'实际阈值：候选 {CONF_THRESHOLD:.2f}，可靠 {RELIABLE_CONF_THRESHOLD:.2f}')
    print(f'ROI MAT: {SAVE_ROI_MAT} | ROI 预览: {SAVE_ROI_PREVIEW}')
    print('实时窗口中按 Q 或 Esc 可提前停止，并正常保存已完成的 AVI/CSV。')
    result_dir = run_inference.main(argv)
    print(f'本次结果目录: {result_dir}')
    return result_dir

## 开始运行

执行下一单元格才会读取 BIN。完整视频在 CPU 上可能需要较长时间。建议使用实时窗口中的 **Q/Esc** 正常结束，不要直接关闭 Notebook 内核，否则正在写入的 AVI 可能不完整。

In [8]:
result_dir = run_project()
result_dir

开始运行 TFW 热成像人脸检测……
实时窗口中按 Q 或 Esc 可提前停止，并正常保存已完成的 AVI/CSV。
BIN 信息：640x512，完整帧 36055，本次抽取 500 帧
提示：文件尾有 655344 字节不足一帧，已安全忽略。
加载模型：D:\major\IRsegementation\IRrecognize\python_code\thermal_face_baseline\weights\yolov5n-face-tfw.pt
运行设备：cpu
Fusing layers... 
主脸关联：候选阈值 0.03，可靠阈值 0.40，最长短缺失 6 帧
AVI 输出：D:\major\IRsegementation\IRrecognize\python_code\thermal_face_baseline\output\bin_full_video_notebook_4\detection_preview.avi，播放帧率 30.000 FPS
[1/500] nir_temp_frame_16uc1.bin frame=15000: 1 个候选，主脸 0.70，推理 126.5 ms
[66/500] nir_temp_frame_16uc1.bin frame=15065: 0 个候选，未关联主脸，推理 123.5 ms
[67/500] nir_temp_frame_16uc1.bin frame=15066: 0 个候选，未关联主脸，推理 146.1 ms
[100/500] nir_temp_frame_16uc1.bin frame=15099: 1 个候选，主脸 0.83，推理 176.3 ms
收到 Q/Esc，提前停止并保存已完成结果。
完成：共处理 100 个输入帧，关联主脸 98 帧，未关联 2 帧；共保留 98 个候选框。
静态图片（missed）：D:\major\IRsegementation\IRrecognize\python_code\thermal_face_baseline\output\bin_full_video_notebook_4\missed，共 2 张
AVI 视频：D:\major\IRsegementation\IRrecognize\python_code\thermal_face_base

WindowsPath('D:/major/IRsegementation/IRrecognize/python_code/thermal_face_baseline/output/bin_full_video_notebook_4')

## 输出内容

- `detection_preview.avi`：包含全部已处理原始帧的人脸框与关键点。
- `missed/`：只保存没有检测到人脸的帧。
- `detections.csv`：逐帧检测框、关键点、置信度和温度统计。
- `summary.json`：本次运行参数、处理帧数和平均耗时。

若输出目录已存在，程序会自动使用 `_2`、`_3` 后缀，不覆盖旧结果。